In [ ]:
# =========================
# Colab Supervised Baseline (Source-only)
# Train: PlantVillage/train (labeled)
# Test : PlantDoc/test (labeled)
# Model: ResNet18 ImageNet pretrained
# Output: best checkpoint + confusion matrix + metrics saved to Google Drive
# =========================
import os, time, json, random, re
from typing import List, Dict, Tuple
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms, models

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
     73,466,894 100%  458.08kB/s    0:02:36 (xfr#236, to-chk=0/264)


In [ ]:
# -------------------------
# A) SOURCE AND TARGET PATHS
# -------------------------

SOURCE_TRAIN_DIR = "/content/drive/MyDrive/dataset/PlantVillage"
TARGET_TEST_DIR  = "/content/drive/MyDrive/dataset/PlantDoc/test"

# Where to save results
OUT_DIR = "/content/drive/MyDrive/plant_disease_project/runs_supervised_baseline"
os.makedirs(OUT_DIR, exist_ok=True)

In [22]:
# -------------------------
# B) Basic configs
# -------------------------
MODEL_NAME = "resnet18"      # "resnet50" also ok
PRETRAINED = True           # ImageNet pretrained
EPOCHS = 10
BATCH_SIZE = 64
LR = 3e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2             # Colab usually ok with 2~4
USE_AMP = True              # mixed precision on CUDA
SEED = 42

In [23]:
# -------------------------
# C) Repro / device
# -------------------------
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("[Info] device =", device)

[Info] device = cuda


In [24]:
# -------------------------
# D) Transforms (ImageNet style)
# -------------------------
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])

train_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    normalize,
])

test_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    normalize,
])

In [ ]:
# -------------------------
# E) Datasets / Loaders
# -------------------------

src_ds = datasets.ImageFolder(SOURCE_TRAIN_DIR, transform=train_tf)
tgt_raw = datasets.ImageFolder(TARGET_TEST_DIR, transform=test_tf)

print("\n[SOURCE classes]", len(src_ds.classes), src_ds.classes)
print("\n[TARGET  classes]", len(tgt_raw.classes), tgt_raw.classes)

num_classes = len(src_ds.classes)


PV2PD = {
    "Pepper__bell___Bacterial_spot": "Bell_pepper leaf spot",
    "Pepper__bell___healthy": "Bell_pepper leaf",

    "Potato___Early_blight": "Potato leaf early blight",
    "Potato___Late_blight": "Potato leaf late blight",
    "Potato___healthy": None,  # PlantDoc does not have potato healthy

    "Tomato_Bacterial_spot": "Tomato leaf bacterial spot",
    "Tomato_Early_blight": "Tomato Early blight leaf",
    "Tomato_Late_blight": "Tomato leaf late blight",
    "Tomato_Leaf_Mold": "Tomato mold leaf",
    "Tomato_Septoria_leaf_spot": "Tomato Septoria leaf spot",
    "Tomato_Spider_mites_Two_spotted_spider_mite": None,  # No spider mites in PlantDoc
    "Tomato__Target_Spot": None,  # No Tomato target spot in PlantDoc
    "Tomato__Tomato_YellowLeaf__Curl_Virus": "Tomato leaf yellow virus",
    "Tomato__Tomato_mosaic_virus": "Tomato leaf mosaic virus",
    "Tomato_healthy": "Tomato leaf",
}

# target folder -> source class idx
src_class_to_idx = src_ds.class_to_idx

pd_folder_to_src_idx: Dict[str, int] = {}
missing = []
for pv_cls, pd_cls in PV2PD.items():
    if pd_cls is None:
        continue
    if pv_cls not in src_class_to_idx:
        missing.append(("PV_missing_in_source", pv_cls))
        continue
    pd_folder_to_src_idx[pd_cls] = src_class_to_idx[pv_cls]

print("\n[Mapping target_folder -> source_idx]")
for k,v in pd_folder_to_src_idx.items():
    print("  ", k, "->", v)

# target test dataset 
class MappedTargetDataset(Dataset):
    def __init__(self, imgfolder: datasets.ImageFolder, folder_to_srcidx: Dict[str,int]):
        self.base = imgfolder
        self.folder_to_srcidx = folder_to_srcidx

        # target
        self.keep_indices = []
        for i, (path, y) in enumerate(self.base.samples):
            folder_name = self.base.classes[y]
            if folder_name in self.folder_to_srcidx:
                self.keep_indices.append(i)

        # statistics
        self.kept_by_class = {}
        for i in self.keep_indices:
            _, y = self.base.samples[i]
            folder_name = self.base.classes[y]
            self.kept_by_class[folder_name] = self.kept_by_class.get(folder_name, 0) + 1

    def __len__(self):
        return len(self.keep_indices)

    def __getitem__(self, idx):
        real_i = self.keep_indices[idx]
        x, y = self.base[real_i]
        folder_name = self.base.classes[y]
        new_y = self.folder_to_srcidx[folder_name]
        return x, new_y

tgt_ds = MappedTargetDataset(tgt_raw, pd_folder_to_src_idx)
print("\n[Target kept samples]", len(tgt_ds))
print("[Kept class counts]", tgt_ds.kept_by_class)

if len(tgt_ds) == 0:
    raise RuntimeError("target test mapping PV2PD and target error")

# DataLoader
src_loader = DataLoader(src_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
tgt_loader = DataLoader(tgt_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)




[SOURCE classes] 15 ['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']

[TARGET  classes] 27 ['Apple Scab Leaf', 'Apple leaf', 'Apple rust leaf', 'Bell_pepper leaf', 'Bell_pepper leaf spot', 'Blueberry leaf', 'Cherry leaf', 'Corn Gray leaf spot', 'Corn leaf blight', 'Corn rust leaf', 'Peach leaf', 'Potato leaf early blight', 'Potato leaf late blight', 'Raspberry leaf', 'Soyabean leaf', 'Squash Powdery mildew leaf', 'Strawberry leaf', 'Tomato Early blight leaf', 'Tomato Septoria leaf spot', 'Tomato leaf', 'Tomato leaf bacterial spot', 'Tomato leaf late blight', 'Tomato leaf mosaic virus', 'Tomato leaf yellow virus', 'Tomato mold leaf',

In [26]:
# -------------------------
# F) Model
# -------------------------
def build_model(model_name: str, num_classes: int, pretrained: bool = True) -> nn.Module:
    model_name = model_name.lower().strip()
    if model_name == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if pretrained else None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        return m
    if model_name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if pretrained else None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        return m
    raise ValueError("Only supports resnet18/resnet50")

model = build_model(MODEL_NAME, num_classes=num_classes, pretrained=PRETRAINED).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(USE_AMP and device.type == "cuda")
)


In [27]:
# -------------------------
# G) Metrics helpers
# -------------------------
@torch.no_grad()
def confusion_matrix(pred: List[int], true: List[int], num_classes: int) -> torch.Tensor:
    cm = torch.zeros((num_classes, num_classes), dtype=torch.long)
    for p, t in zip(pred, true):
        cm[t, p] += 1
    return cm

def macro_f1_from_cm(cm: torch.Tensor) -> float:
    # cm: [true, pred]
    f1s = []
    for c in range(cm.size(0)):
        tp = cm[c, c].item()
        fp = cm[:, c].sum().item() - tp
        fn = cm[c, :].sum().item() - tp

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
        f1s.append(f1)
    return float(sum(f1s) / len(f1s)) if f1s else 0.0


In [28]:
# -------------------------
# H) Train / Eval
# -------------------------
def train_one_epoch() -> Dict[str, float]:
    model.train()
    total_loss, total_correct, total_count = 0.0, 0, 0
    start = time.time()

    for images, labels in src_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type="cuda",enabled=(USE_AMP and device.type == "cuda")):
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_count += images.size(0)

    return {
        "loss": total_loss / max(1, total_count),
        "acc": total_correct / max(1, total_count),
        "sec": time.time() - start
    }

@torch.no_grad()
def evaluate() -> Dict[str, object]:
    model.eval()
    total_loss, total_correct, total_count = 0.0, 0, 0
    all_preds, all_true = [], []

    for images, labels in tgt_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(images)
        loss = criterion(logits, labels)
        preds = logits.argmax(dim=1)

        total_loss += loss.item() * images.size(0)
        total_correct += (preds == labels).sum().item()
        total_count += images.size(0)

        all_preds.extend(preds.cpu().tolist())
        all_true.extend(labels.cpu().tolist())

    cm = confusion_matrix(all_preds, all_true, num_classes=num_classes)
    macro_f1 = macro_f1_from_cm(cm)

    return {
        "loss": total_loss / max(1, total_count),
        "acc": total_correct / max(1, total_count),
        "macro_f1": macro_f1,
        "cm": cm
    }

In [ ]:
# -------------------------
# I) Run training
# -------------------------

config = {
    "SOURCE_TRAIN_DIR": SOURCE_TRAIN_DIR,
    "TARGET_TEST_DIR": TARGET_TEST_DIR,
    "OUT_DIR": OUT_DIR,
    "MODEL_NAME": MODEL_NAME,
    "PRETRAINED": PRETRAINED,
    "EPOCHS": EPOCHS,
    "BATCH_SIZE": BATCH_SIZE,
    "LR": LR,
    "WEIGHT_DECAY": WEIGHT_DECAY,
    "NUM_WORKERS": NUM_WORKERS,
    "USE_AMP": USE_AMP,
    "SEED": SEED,
    "classes": src_ds.classes
}
with open(os.path.join(OUT_DIR, "config.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

best_acc = -1.0
best_path = os.path.join(OUT_DIR, "best.pt")

for epoch in range(1, EPOCHS + 1):
    tr = train_one_epoch()
    ev = evaluate()

    print(
        f"Epoch {epoch:03d}/{EPOCHS} | "
        f"train loss {tr['loss']:.4f} acc {tr['acc']:.4f} ({tr['sec']:.1f}s) | "
        f"PlantDoc test loss {ev['loss']:.4f} acc {ev['acc']:.4f} macroF1 {ev['macro_f1']:.4f}"
    )

    if ev["acc"] > best_acc:
        best_acc = ev["acc"]
        torch.save({
            "model_state_dict": model.state_dict(),
            "epoch": epoch,
            "best_acc": best_acc,
            "classes": src_ds.classes,
            "model_name": MODEL_NAME,
        }, best_path)

Epoch 001/10 | train loss 0.2088 acc 0.9358 (123.6s) | PlantDoc test loss 4.3815 acc 0.3039 macroF1 0.1936
Epoch 002/10 | train loss 0.0819 acc 0.9732 (119.5s) | PlantDoc test loss 5.8322 acc 0.2353 macroF1 0.1396
Epoch 003/10 | train loss 0.0569 acc 0.9813 (117.3s) | PlantDoc test loss 5.4344 acc 0.2157 macroF1 0.1103
Epoch 004/10 | train loss 0.0483 acc 0.9840 (117.0s) | PlantDoc test loss 4.9719 acc 0.2353 macroF1 0.1385
Epoch 005/10 | train loss 0.0382 acc 0.9866 (117.4s) | PlantDoc test loss 5.6893 acc 0.2059 macroF1 0.1364
Epoch 006/10 | train loss 0.0387 acc 0.9873 (119.7s) | PlantDoc test loss 4.8056 acc 0.2745 macroF1 0.1779
Epoch 007/10 | train loss 0.0289 acc 0.9910 (118.8s) | PlantDoc test loss 6.0694 acc 0.2255 macroF1 0.1392
Epoch 008/10 | train loss 0.0301 acc 0.9903 (119.7s) | PlantDoc test loss 6.7768 acc 0.2255 macroF1 0.1160
Epoch 009/10 | train loss 0.0253 acc 0.9910 (119.9s) | PlantDoc test loss 7.3297 acc 0.2059 macroF1 0.1104
Epoch 010/10 | train loss 0.0298 acc 

In [30]:
# final dump cm
final = evaluate()
cm_path = os.path.join(OUT_DIR, "confusion_matrix.txt")
with open(cm_path, "w", encoding="utf-8") as f:
    f.write("Confusion Matrix (rows=true, cols=pred)\n")
    f.write(str(final["cm"].tolist()) + "\n")
    f.write(f"Final Acc: {final['acc']:.6f}\n")
    f.write(f"Final Macro-F1: {final['macro_f1']:.6f}\n")

print("\n[Done] best_acc =", best_acc)
print("[Saved] best checkpoint:", best_path)
print("[Saved] confusion matrix:", cm_path)


[Done] best_acc = 0.30392156862745096
[Saved] best checkpoint: /content/drive/MyDrive/plant_disease_project/runs_supervised_baseline/best.pt
[Saved] confusion matrix: /content/drive/MyDrive/plant_disease_project/runs_supervised_baseline/confusion_matrix.txt
